# Factor research on the H5 store with `factor_common`

End-to-end usage of the common factor framework: build a `FactorManager`
over the point-in-time `crypto_quant.h5` store, evaluate a pure `.py`
factor, reload the persisted Parquet artifacts, inspect cost/quality
diagnostics, and render the HTML report.

Reads against the H5 store are read-only; all runs and reports are written
under `data/factor_results/` and `reports/` by default, or under
`FACTOR_COMMON_OUTPUT_DIR` when that environment variable is set (the
acceptance script `scripts/verify_factor_common.py` sets it together with
`FACTOR_COMMON_H5`, `FACTOR_COMMON_START`, and `FACTOR_COMMON_END`).
See `docs/factor_common.md` for the full contract.

## 1. Repository path setup

The repository root is discovered with `pathlib` from an injected
`FACTOR_COMMON_REPO_ROOT`, the notebook path, or the current working
directory. This lets execution work from an unrelated working directory
without user-specific absolute paths.

In [ ]:
from pathlib import Path
import os
import sys


NOTEBOOK_PATH = (
    Path(os.environ["FACTOR_COMMON_NOTEBOOK"]).expanduser().resolve()
    if os.environ.get("FACTOR_COMMON_NOTEBOOK")
    else None
)
INJECTED_ROOT = os.environ.get("FACTOR_COMMON_REPO_ROOT")


def _find_repo_root(start: Path, notebook_path: Path | None = None) -> Path:
    candidates = []
    if INJECTED_ROOT:
        candidates.append(Path(INJECTED_ROOT).expanduser().resolve())
    candidates.extend([start.resolve(), *start.resolve().parents])
    if notebook_path is not None:
        candidates.extend([notebook_path.parent, *notebook_path.parent.parents])
    for candidate in candidates:
        if (candidate / "factor_common").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(f"cannot locate the repository root from {start}")


ROOT = _find_repo_root(Path.cwd(), NOTEBOOK_PATH)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

H5_PATH = Path(os.environ.get("FACTOR_COMMON_H5", ROOT / "data" / "crypto_quant.h5"))
OUTPUT_DIR = os.environ.get("FACTOR_COMMON_OUTPUT_DIR")
START = os.environ.get("FACTOR_COMMON_START")  # e.g. "2024-02-01"
END = os.environ.get("FACTOR_COMMON_END")      # e.g. "2024-03-15"
print(f"repo root: {ROOT}")
print(f"H5 store : {H5_PATH} (exists={H5_PATH.is_file()})")

## 2. Manager construction

`FactorManager` wires loading, value computation, storage, accounting,
metrics, and rendering. Explicit `h5_path`/`base_dir`/`reports_dir` keep
every artifact of this run under one directory.

In [ ]:
from factor_common import FactorManager

artifact_dirs = {}
if OUTPUT_DIR:
    out = Path(OUTPUT_DIR)
    artifact_dirs = {"base_dir": out / "factor_results", "reports_dir": out / "reports"}
manager = FactorManager(h5_path=H5_PATH, **artifact_dirs)
print(f"h5: {manager.h5_path}")
print(f"artifacts: {manager.base_dir}")
print(f"reports:   {manager.reports_dir}")

## 3. Available fields and covered range

The point-in-time `DataProvider` lists the daily market fields, the store's
covered time range, and the known symbols.

In [ ]:
dp = manager.dp
print("fields:", dp.list_datas())
first, last = dp.get_time_range()
print(f"time range: {first.date()} .. {last.date()}")
print(f"symbols: {len(dp.symbols)}")

## 4. Factor template syntax

A factor is a pure `.py` module: `TYPE`, `META`, `SETTING`, and
`calc_factor(data_ctx)`. `manager.create_template("my_factor")` scaffolds a
new file from this template without ever overwriting an existing one.

In [ ]:
template_path = ROOT / "factor_common" / "templates" / "regular_daily.py.tmpl"
print(template_path.read_text(encoding="utf-8"))

## 5. Evaluate the `.py` example factor

`example_momentum` computes N-day log momentum. Evaluation computes and
persists the factor values *before* any funding/execution data is touched,
runs the static future-leak scan and cutoff replays, then accounts the
gross / trading-net / all-costs scenarios and renders the report. Data
limitations surface as `status="incomplete"` (or `"insufficient_data"`),
never as silently zero-filled numbers.

In [ ]:
EXAMPLE = ROOT / "factor_analyse" / "factor_mining" / "example_momentum.py"
params = {"rebalance_days": 1}
if START:
    params["start"] = START
if END:
    params["end"] = END
result = manager.evaluate(str(EXAMPLE), params=params, plot=True)
print("status:", result["status"])
print("run_id:", result["run_id"], " evaluation_id:", result["evaluation_id"])
print("factor value matrix:", result["factor_value"].shape)
print("report:", result["paths"]["report_path"])

## 6. Saved factor access

Values persist as a long-format `factor.parquet` (`date`, `instrument`,
`factor`) plus axis metadata; `get_value` rebuilds the exact matrix.

In [ ]:
reloaded = manager.get_value("example_momentum", run_id=result["run_id"])
print("reload equals computed matrix:", reloaded.equals(result["factor_value"]))
last_day = reloaded.index[-1]
print(f"last signal date: {last_day.date()}  valid names: {int(reloaded.iloc[-1].notna().sum())}")

## 7. Full-sample performance

`factor_performance["samples"]["full"]` holds IC-family metrics against the
evaluation-only next-open labels; `factor_performance["scenarios"][<name>]
["full"]` holds the per-scenario portfolio metrics. Incomplete scenarios
report `null` full-sample metrics (plus a separately labeled
`known_segment`), never zero-filled numbers.

In [ ]:
performance = result["factor_performance"]
full_ic = performance["samples"]["full"]["ic"]
ic_keys = ("ic_mean", "rank_ic_mean", "icir", "annualized_icir",
           "t_stat", "p_value", "n_dates")
print("IC (full sample):", {key: full_ic[key] for key in ic_keys})
for name, block in performance["scenarios"].items():
    full = block["full"]
    if full is None:
        print(f"{name}: status={block['status']} -> full-sample metrics are null")
    else:
        metric_keys = ("total_return", "annual_return", "sharpe",
                       "max_drawdown", "turnover", "n_periods")
        print(f"{name}:", {key: full[key] for key in metric_keys})

## 8. Cost and quality diagnostics

Every ledger labels its cost cash flows (`fee`, `funding_cashflow`); the
all-costs scenario additionally records per-day funding coverage. Coverage
backed only by observed events is `unknown` — never silently complete.

In [ ]:
for name, scenario in result["factor_result"]["scenarios"].items():
    ledger = scenario["ledger"]
    fees = round(float(ledger["fee"].sum()), 8) if not ledger.empty else None
    funding = round(float(ledger["funding_cashflow"].sum()), 8) if not ledger.empty else None
    print(f"{name}: status={scenario['status']} fees={fees} "
          f"funding_cashflow={funding} halt={scenario['diagnostics']['halt_reason']}")
print("funding coverage:", result["diagnostics"]["coverage"]["funding"]["status_counts"])
validation = result["diagnostics"]["validation"]
print("static scan:", validation["static_scan"]["status"])
print("cutoff replay:", validation["cutoff"]["status"],
      [entry["max_abs_diff"] for entry in validation["cutoff"].get("cutoffs", [])])

## 9. Report rendering

Rendering is a pure display transform over saved result tables. The report
labels every cost scenario and states data limitations explicitly
(incomplete segments, unknown funding coverage, missing benchmark).

In [ ]:
info = manager.plot_result(result)
print("report written:", info["output_path"])
print("display window:", info["start"], "..", info["end"])
print("warnings:", info["warnings"])

## 10. External matrices are unverified

A precomputed external DataFrame can be evaluated through the same chain,
but it carries no source code to scan and no recomputable history to
replay, so its cutoff validation is recorded as `not_verified` — an
explicit limitation, not a passed check.

In [ ]:
import numpy as np
import pandas as pd

values = result["factor_value"]
dates = pd.date_range(values.index[-5], values.index[-1], freq="D", name="date")
instruments = list(values.columns[:12])
external = pd.DataFrame(
    np.arange(len(dates) * len(instruments), dtype="float64").reshape(len(dates), len(instruments)),
    index=dates,
    columns=instruments,
)
external_result = manager.evaluate(
    external,
    factor_name="external_demo",
    params={"rebalance_days": 1, "n_groups": 3, "include_funding": False},
    plot=False,
)
print("external status:", external_result["status"])
cutoff = external_result["diagnostics"]["validation"]["cutoff"]
print("cutoff validation:", cutoff["status"], "-", cutoff.get("reason"))